# Capítulo 1 — Introducción a las Pruebas de Extremo a Extremo (E2E)

**Objetivo:** Comprender qué son las pruebas E2E, por qué ocupan la cima de la pirámide de pruebas, cuándo son apropiadas (y cuándo no), y conocer el panorama actual de herramientas.

---

## 1. ¿Qué es una prueba E2E?

Una **prueba de extremo a extremo** (End-to-End, E2E) simula el comportamiento de un **usuario real** interactuando con el sistema completo: desde la interfaz de usuario hasta la base de datos, pasando por toda la lógica de negocio, la red y los servicios externos.

A diferencia de las pruebas unitarias (que aíslan un módulo) o las de integración (que prueban la colaboración entre módulos), las pruebas E2E **no usan ningún sustituto**: todo es real.

> «Las pruebas E2E verifican que los flujos de usuario completos funcionan correctamente, desde el primer clic hasta el último efecto en la base de datos.»

### Ejemplo concreto

En nuestro gestor de tareas, un flujo E2E típico sería:

```
1. El usuario abre el navegador en http://localhost:5000
2. Escribe "Comprar pan" en el campo de texto
3. Hace clic en "Agregar"
4. Verifica que "Comprar pan" aparece en la lista
5. Hace clic en "Completar"
6. Verifica que la tarea muestra el badge "✓ Completada"
7. Hace clic en "Eliminar"
8. Verifica que "Comprar pan" ya no está en la lista
```

Cada uno de estos pasos involucra la UI (HTML/CSS), el servidor Flask, el modelo de datos y el archivo JSON — **todo al mismo tiempo**.

---

## 2. La pirámide de pruebas revisitada

Recordemos la pirámide de pruebas (Mike Cohn, 2009) y ubiquemos las pruebas E2E:

```
              ╔══════════╗
             ╔╝   E2E    ╚╗      ← Pocos tests. Lentos. Costosos. Alta confianza.
            ╔╝  (UI/Web)  ╚╗
           ╔══════════════╗
          ╔╝  Integration  ╚╗    ← Moderados. Velocidad media.
         ╔╝   (API/módulos) ╚╗
        ╔══════════════════╗
       ╔╝      Unit Tests   ╚╗   ← Muchos. Rápidos. Baratos.
      ╔╝  (funciones/clases) ╚╗
     ╚══════════════════════╝
```

### ¿Por qué pocos tests E2E?

| Factor | Unit | Integration | E2E |
|--------|------|-------------|-----|
| **Velocidad** | Milisegundos | Segundos | Segundos/minutos |
| **Costo de mantenimiento** | Bajo | Medio | Alto |
| **Estabilidad (flakiness)** | Muy estable | Estable | Inestable (por naturaleza) |
| **Diagnóstico de fallos** | Fácil | Medio | Difícil |
| **Confianza del sistema completo** | Baja | Media | Alta |
| **Infraestructura requerida** | Ninguna | Parcial | Completa |

La regla práctica: **muchos tests unitarios, algunos de integración, pocos E2E**. Los E2E son valiosos precisamente por su escasez — cubren los flujos más críticos del negocio.

---

## 3. ¿Qué detectan las pruebas E2E que las demás no detectan?

Hay categorías de fallos que solo son visibles desde el nivel E2E:

| Tipo de fallo | Descripción | Ejemplo |
|---------------|-------------|------|
| **Fallo de renderizado** | El servidor devuelve datos correctos, pero la plantilla HTML los muestra mal | El título aparece escapado como `&lt;script&gt;` |
| **Routing incorrecto** | Una redirección envía al usuario a la página equivocada | Tras crear una tarea, redirige a `/error` |
| **JavaScript roto** | Un script del frontend impide que el formulario funcione | El botón no responde por un error JS |
| **CSS que oculta elementos** | Un elemento existe en el DOM pero es invisible para el usuario | El botón 'Completar' tiene `display:none` |
| **Interacción compleja** | Flujos que requieren múltiples pasos encadenados | Crear → completar → verificar en una sola sesión |
| **Estado de sesión** | Problemas con cookies, autenticación o localStorage | Tras login, la sesión expira incorrectamente |

---

## 4. Cuándo usar pruebas E2E

### ✅ Casos donde E2E es la herramienta correcta

- **Flujos críticos del negocio**: el proceso de checkout en un e-commerce, el registro de usuario, el proceso de pago.
- **Regresión de features prioritarias**: asegurarse de que las funcionalidades más usadas no se rompen con cada deploy.
- **Integraciones entre sistemas heterogéneos**: cuando el frontend usa un framework diferente al backend.
- **Validación de experiencia de usuario**: verificar que la UI responde correctamente a las acciones del usuario.

### ❌ Casos donde E2E NO es la herramienta correcta

- **Lógica de negocio compleja**: usar tests unitarios — son más rápidos y el diagnóstico es más fácil.
- **Validación de contratos entre APIs**: usar tests de integración.
- **Casos extremos de datos**: probar 50 combinaciones de entrada — demasiado lento en E2E.
- **Errores de manejo de excepciones internas**: los tests unitarios son más precisos.

> 💡 **Regla de oro:** si puedes probar algo con una prueba unitaria o de integración, hazlo así. Reserva los E2E para lo que **solo puede ser probado desde el navegador**.

---

## 5. Panorama de herramientas E2E

El ecosistema de herramientas para pruebas E2E es amplio. Aquí las principales:

| Herramienta | Lenguaje | Protocolo | Año | Notas |
|-------------|----------|-----------|-----|-------|
| **Selenium** | Multi (Python, Java, JS...) | WebDriver (W3C) | 2004 | El pionero. Muy maduro. |
| **Playwright** | Python, JS, Java, .NET | CDP + WebSocket | 2020 | Moderno, rápido, sin flakiness. |
| **Cypress** | JavaScript | Interno (no WebDriver) | 2017 | Excelente DX en JS. Solo frontend. |
| **Puppeteer** | JavaScript | CDP | 2017 | Solo Chromium. Bajo nivel. |
| **Appium** | Multi | WebDriver (mobile) | 2012 | Para apps móviles. |

En este taller usaremos **Playwright** como herramienta principal y **Selenium** como referencia histórica y comparativa (ver Capítulo 5).

---

## 6. ¿Cómo funcionan las herramientas E2E? El protocolo WebDriver vs CDP

Entender el protocolo es importante para comprender las diferencias de comportamiento entre herramientas.

### WebDriver (W3C Standard)

```
Test Code  →  WebDriver Client  →  [HTTP REST API]  →  WebDriver Server  →  Browser
             (selenium, etc.)                          (chromedriver, etc.)
```

- Cada acción (`click`, `fill`, etc.) es una petición HTTP al servidor WebDriver.
- El servidor WebDriver controla el browser mediante su protocolo nativo.
- **Ventaja:** estándar W3C, funciona con cualquier browser.
- **Desventaja:** latencia por el ciclo HTTP, limitaciones en intercepción de red.

### Chrome DevTools Protocol (CDP)

```
Test Code  →  Playwright/Puppeteer  →  [WebSocket]  →  Browser (Chrome/Firefox/Safari)
```

- Comunicación directa vía WebSocket — mucho más rápida.
- Acceso completo a las DevTools: intercepción de red, emulación de dispositivos, trazas.
- **Ventaja:** velocidad, capacidades avanzadas, auto-esperas incorporadas.
- **Desventaja:** históricamente solo Chromium (Playwright lo extendió a Firefox y Safari).

---

## 7. El sistema bajo prueba: arquitectura

En este taller trabajaremos con una aplicación Flask de gestión de tareas:

```
┌─────────────────────────────────────────┐
│          Navegador (Playwright)          │  ← El test automatiza esto
│   Renderiza HTML, ejecuta formularios    │
└──────────────────┬──────────────────────┘
                   │ HTTP GET/POST
┌──────────────────▼──────────────────────┐
│           Flask App (app.py)             │  ← Rutas web
│  GET /          → index.html            │
│  POST /tasks    → crear tarea           │
│  POST /tasks/<id>/complete → completar  │
│  POST /tasks/<id>/delete   → eliminar   │
└──────────────────┬──────────────────────┘
                   │ llama a
┌──────────────────▼──────────────────────┐
│         TaskRepository (models.py)       │  ← Lógica de datos
└──────────────────┬──────────────────────┘
                   │ lee/escribe
              [tasks.json]
```

Una prueba E2E de este sistema atraviesa **todas** las capas del diagrama en cada operación.

---

## 8. Ejercicio de reflexión

In [ ]:
# Exploración del sistema bajo prueba
# Asegúrate de estar en la carpeta e2e_testing/
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import inspect
from src.app import app
from src.models import TaskRepository, Task

# Inspeccionamos las rutas de la aplicación Flask
print("=== Rutas de la aplicación Flask ===")
for rule in app.url_map.iter_rules():
    methods = ", ".join(sorted(rule.methods - {"HEAD", "OPTIONS"}))
    print(f"  [{methods:8}]  {rule.rule}")

print()
print("=== Métodos públicos de TaskRepository ===")
for name, method in inspect.getmembers(TaskRepository, predicate=inspect.isfunction):
    if not name.startswith('_'):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

### Preguntas para el informe

1. **Clasificación de pruebas:** Para cada uno de los siguientes escenarios de prueba, indica si lo harías con una prueba unitaria, de integración o E2E. Justifica tu respuesta:
   - Verificar que `calcular_promedio([])` retorna `None`.
   - Verificar que al crear una tarea desde el formulario web, aparece en la lista de la página.
   - Verificar que `TaskRepository.add()` rechaza títulos vacíos.
   - Verificar que la ruta `/tasks` redirige a `/` después de crear una tarea.
   - Verificar que el botón 'Completar' desaparece después de completar una tarea.

2. **Costo de mantenimiento:** ¿Por qué los tests E2E tienen mayor costo de mantenimiento que los unitarios? Da un ejemplo concreto de qué tipo de cambio en la aplicación rompería un test E2E pero no un test unitario.

3. **Flujos críticos:** Si solo pudieras escribir **3 tests E2E** para el gestor de tareas, ¿cuáles elegirías? ¿Por qué esos y no otros?

4. **Protocolo:** ¿Por qué el uso de WebSockets (CDP) hace que Playwright sea más rápido que Selenium con WebDriver HTTP? ¿Qué implicaciones tiene esto para suites de tests grandes?